# PGBM Simulation: Phase-wise Colab Notebook
## Phase 1 - Experimental Contract

This phase defines units, configuration, deterministic random seeds, and the common results schema.

In [ ]:
from dataclasses import asdict, dataclass
from pathlib import Path
import csv
import hashlib
import json
from typing import Dict, Tuple

OUTPUT_DIR = Path('/content/pgbm_phase1_results')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Output directory:', OUTPUT_DIR)

## 1. Configuration

These values are the experimental contract. Later phases must read these values instead of silently introducing different units or assumptions.

In [ ]:
@dataclass(frozen=True)
class ExperimentConfig:
    schema_version: str = 'phase1.v1'
    experiment_name: str = 'pgbm_initial_contract'
    base_seed: int = 20260921
    time_unit: str = 'minute'
    distance_unit: str = 'meter'
    energy_unit: str = 'joule'
    payload_unit: str = 'kilogram'
    workspace_bounds: Tuple[float, float, float, float, float, float] = (0., 0., 0., 100., 100., 60.)
    planning_time: float = 0.0
    decay_rate: float = 0.02
    task_count: int = 6
    uav_count: int = 2
    reserve_fraction: float = 0.2

    def validate(self):
        assert len(self.workspace_bounds) == 6
        assert all(self.workspace_bounds[i] < self.workspace_bounds[i + 3] for i in range(3))
        assert self.base_seed >= 0
        assert self.planning_time >= 0
        assert self.decay_rate > 0
        assert self.task_count > 0 and self.uav_count > 0
        assert 0 <= self.reserve_fraction < 1

config = ExperimentConfig()
config.validate()
config_dict = asdict(config)
config_dict['workspace_bounds'] = list(config.workspace_bounds)
display(config_dict)

## 2. Deterministic component seeds

Each component receives a stable seed derived from the same base seed. This lets us change task generation without silently changing the environment generation.

In [ ]:
def seed_for(component: str, repetition: int = 0) -> int:
    key = f'{config.base_seed}:{component}:{repetition}'.encode('utf-8')
    return int.from_bytes(hashlib.sha256(key).digest()[:8], 'big') % (2**32)

components = ['environment', 'tasks', 'uavs', 'planner', 'recourse']
component_seeds = {name: seed_for(name) for name in components}
display(component_seeds)
assert component_seeds == {name: seed_for(name) for name in components}
assert len(set(component_seeds.values())) == len(components)

## 3. Standard result schema

Every later planner, baseline, and experiment will write these fields. Empty values are acceptable in Phase 1 because no mission has been executed yet.

In [ ]:
metric_fields = [
    'schema_version', 'experiment_name', 'seed', 'method', 'scenario_id', 'status',
    'objective_value', 'served_tasks', 'deferred_tasks', 'average_completion_delay',
    'total_distance', 'total_travel_time', 'total_energy', 'payload_utilization',
    'accepted_replacements', 'rejected_replacements', 'safe_return_rate', 'runtime_seconds'
]
print('Metric count:', len(metric_fields))
print('Metric fields:', metric_fields)

## 4. Save the Phase 1 manifest

This manifest is the reproducibility record for the phase.

In [ ]:
manifest = {
    'config': config_dict,
    'component_seeds': component_seeds,
    'metric_fields': metric_fields,
}
manifest_path = OUTPUT_DIR / 'phase1_manifest.json'
manifest_path.write_text(json.dumps(manifest, indent=2))

metrics_path = OUTPUT_DIR / 'phase1_metrics.csv'
with metrics_path.open('w', newline='') as handle:
    csv.writer(handle).writerow(metric_fields)

print('Saved:', manifest_path)
print('Saved:', metrics_path)
print('Phase 1 complete. Stop here and review the contract before Phase 2.')